# Kedi & Köpek Irkı Sınıflandırma — ResNet-18 + Oxford-IIIT Pet

**Zeynep Özge Temel - 23120205072**

Colab'da çalıştırmak için: önce **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU** seçin, sonra hücreleri yukarıdan aşağıya sırayla çalıştırın (her hücrenin solundaki ▶ butonu).

## 0. Kurulum
Repo klonlanır ve gerekli kütüphaneler kurulur.

In [ ]:
!git clone https://github.com/zeynepozgetemel/kedi-kopek-irk-siniflandirma.git
%cd kedi-kopek-irk-siniflandirma
!pip -q install timm gradio

## 1. Veri setini indir
Oxford-IIIT Pet: 37 ırk (12 kedi, 25 köpek), 7.349 görüntü. Resmi `trainval` kümesi %85 eğitim / %15 doğrulama olarak bölünür; resmi `test` kümesi test olarak kullanılır.

In [ ]:
!python download_data.py

## 2. Eğitim
ResNet-18 (ImageNet ön-eğitimli) → son katman 37 sınıf. AdamW, OneCycle LR, CrossEntropy (label smoothing 0.1), batch 32, 8 epoch, 224×224.

In [ ]:
!python train.py --data_dir data/data --epochs 8

## 3. Sonuçlar

In [ ]:
import json
m = json.load(open('outputs/metrics.json'))
{k: v for k, v in m.items() if k not in ('history', 'args')}

In [ ]:
from IPython.display import Image, display
display(Image('outputs/training_curves.png'))
display(Image('outputs/confusion_matrix.png', width=800))
print(open('outputs/classification_report.txt').read())

## 4. Kendi fotoğraflarımla test
Telefonla çekilen fotoğraflar `my_photos/` klasöründe.

In [ ]:
!python predict_photos.py my_photos/IMG_6621.jpg my_photos/IMG_7152.jpg my_photos/IMG_2152.jpg my_photos/IMG_6644.jpg my_photos/IMG_3050.jpg --truth "british_shorthair,persian,sokak kedisi (karisik),sokak kedisi (karisik),siluet - belirsiz"
display(Image('outputs/my_photos_results.png', width=900))

### Ek deney: aynı ırk, farklı çekim koşulları

In [ ]:
!python experiment_same_breed.py
display(Image('outputs/experiment_same_breed.png', width=900))

## 5. Gradio uygulaması
Çıkan **public URL** linkine tıklayınca uygulama açılır; fotoğraf yükleyip tahmini görebilirsiniz.

In [ ]:
import app
app.demo.launch(share=True)